Illustris API request function

In [1]:
import requests

baseUrl = 'http://www.tng-project.org/api/'
headers = {"api-key":"c489f7f5a9e0e61f5060fd16e9fb975e"}

data_dir = './illustris_data/'

def get(path, params=None):
    # make HTTP GET request to path
    r = requests.get(path, params=params, headers=headers)

    # raise exception if response code is not HTTP SUCCESS (200)
    r.raise_for_status()

    if r.headers['content-type'] == 'application/json':
        return r.json() # parse json responses automatically

    if 'content-disposition' in r.headers:
        filename = data_dir + r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        return filename # return the filename string

    return r

## Pick primary halo

In [2]:
sim = get(baseUrl+'TNG50-1/')
sim.keys()

dict_keys(['name', 'description', 'name_alt', 'boxsize', 'z_start', 'z_final', 'cosmology', 'omega_0', 'omega_L', 'omega_B', 'hubble', 'physics_model', 'has_cooling', 'has_starformation', 'has_winds', 'has_blackholes', 'mass_gas', 'mass_dm', 'softening_dm_comoving', 'softening_stars_comoving', 'softening_blackholes_comoving', 'softening_gas_comoving', 'softening_dm_max_phys', 'softening_stars_max_phys', 'softening_blackholes_max_phys', 'softening_gas_max_phys', 'softening_gas_factor', 'softening_gas_comoving_min', 'num_dm', 'num_tr_mc', 'num_tr_vel', 'longids', 'is_uniform', 'is_zoom', 'is_subbox', 'num_files_snapshot', 'num_files_groupcat', 'num_files_rockstar', 'num_files_lhalotree', 'num_files_sublink', 'num_files_ctrees', 'filesize_lhalotree', 'filesize_sublink', 'filesize_ctrees', 'filesize_ics', 'filesize_simulation', 'has_fof', 'has_subfind', 'has_rockstar', 'has_lhalotree', 'has_sublink', 'has_ctrees', 'permission_required', 'num_snapshots', 'url', 'parent_simulation', 'child_s

In [3]:
snaps = get(sim['snapshots'])

In [4]:
snaps[-1] # latest snapshot, z=0

{'number': 99,
 'redshift': 2.22044604925031e-16,
 'num_groups_subfind': 5688113,
 'url': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/'}

In [5]:
z0_halo = get(snaps[-1]['url']+'halos/0/') # first available primary halo; can iterate over these to find MW-like subjects
z0_halo

{'snap': 99,
 'halo_id': 0,
 'meta': {'simulation': 'http://www.tng-project.org/api/TNG50-1/',
  'snapshot': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/',
  'url': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/halos/0/',
  'info': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/halos/0/info.json'},
 'vis': {'halo_gas_dens': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/halos/0/vis.png?partType=gas',
  'halo_gas_temp': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/halos/0/vis.png?partType=gas&partField=temp',
  'halo_dm_dens': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/halos/0/vis.png?partType=dm',
  'halo_stellar_dens': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/halos/0/vis.png?partType=stars',
  'galaxy_gas_dens': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/halos/0/vis.png?partType=gas&size=5.0&sizeType=rHalfMassStars',
  'galaxy_gas_dens_faceon': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/halos/0/vis.png?partT

Get all subhalos in this group

In [6]:
subhalos = get(z0_halo['meta']['url'],{'limit':z0_halo['child_subhalos']['count']}) # this can take a while to run, depending on subhalo count

In [7]:
subhalo_list = subhalos['child_subhalos']['results']

In [8]:
# examine one specific subhalo
subhalo = get(subhalo_list[0]['url'])
subhalo

{'snap': 99,
 'id': 0,
 'bhmdot': 0.000185235,
 'cm_x': 7300.43,
 'cm_y': 24514.0,
 'cm_z': 21300.2,
 'gasmetallicity': 0.00954238,
 'gasmetallicityhalfrad': 0.011177,
 'gasmetallicitymaxrad': 0.0,
 'gasmetallicitysfr': 0.00758403,
 'gasmetallicitysfrweighted': 0.00803236,
 'pos_x': 7307.24,
 'pos_y': 24550.4,
 'pos_z': 21302.6,
 'halfmassrad': 362.23,
 'halfmassrad_gas': 383.195,
 'halfmassrad_dm': 371.711,
 'halfmassrad_stars': 29.2812,
 'halfmassrad_bhs': 0.0,
 'len': 718577369,
 'len_gas': 254750400,
 'len_dm': 369615124,
 'len_stars': 94211842,
 'len_bhs': 3,
 'mass': 13289.6,
 'mass_gas': 1559.96,
 'mass_dm': 11360.8,
 'mass_stars': 368.293,
 'mass_bhs': 0.579029,
 'massinhalfrad': 407.339,
 'massinhalfrad_gas': 3.09503,
 'massinhalfrad_dm': 219.518,
 'massinhalfrad_stars': 184.146,
 'massinhalfrad_bhs': 0.578949,
 'massinmaxrad': 0.001185,
 'massinmaxrad_gas': 0.0,
 'massinmaxrad_dm': 3.1e-05,
 'massinmaxrad_stars': 0.001154,
 'massinmaxrad_bhs': 0.0,
 'massinrad': 894.026,
 'ma

Read progenitor branch to see merger history

In [9]:
import h5py

prog_branch_data = get(subhalo['trees']['sublink_mpb'])

prog_branch = h5py.File(prog_branch_data, 'r')

In [ ]:
print(prog_branch.keys())

<KeysViewHDF5 ['DescendantID', 'FirstProgenitorID', 'FirstSubhaloInFOFGroupID', 'GroupBHMass', 'GroupBHMdot', 'GroupCM', 'GroupFirstSub', 'GroupGasMetalFractions', 'GroupGasMetallicity', 'GroupLen', 'GroupLenType', 'GroupMass', 'GroupMassType', 'GroupNsubs', 'GroupPos', 'GroupSFR', 'GroupStarMetalFractions', 'GroupStarMetallicity', 'GroupVel', 'GroupWindMass', 'Group_M_Crit200', 'Group_M_Crit500', 'Group_M_Mean200', 'Group_M_TopHat200', 'Group_R_Crit200', 'Group_R_Crit500', 'Group_R_Mean200', 'Group_R_TopHat200', 'LastProgenitorID', 'MainLeafProgenitorID', 'Mass', 'MassHistory', 'NextProgenitorID', 'NextSubhaloInFOFGroupID', 'NumParticles', 'RootDescendantID', 'SnapNum', 'SubfindID', 'SubhaloBHMass', 'SubhaloBHMdot', 'SubhaloCM', 'SubhaloGasMetalFractions', 'SubhaloGasMetalFractionsHalfRad', 'SubhaloGasMetalFractionsMaxRad', 'SubhaloGasMetalFractionsSfr', 'SubhaloGasMetalFractionsSfrWeighted', 'SubhaloGasMetallicity', 'SubhaloGasMetallicityHalfRad', 'SubhaloGasMetallicityMaxRad', 'Subh

In [11]:
# get ID and position of earliest progenitor
print(f"z=20 progenitor: subhalo {prog_branch['SubfindID'][-1]} at position {prog_branch['SubhaloPos'][-1]}")

z=20 progenitor: subhalo 1171 at position [ 6910.7217 23544.307  20854.303 ]
